# 09k Alljoined-1.6M, converted to the THINGS-EEG2 layout

**Kaggle:** CPU, Internet on. **Inputs:** the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g, for the
image order). **Secret:** `HF_TOKEN`, a Hugging Face read token of an account that accepted the Alljoined terms.

Alljoined-1.6M (Xu et al. 2025) recorded 20 people with a 32-channel consumer headset viewing the THINGS-EEG2 images
in the same paradigm. For each person this notebook downloads the authors' preprocessed files, keeps the 17 channels
THINGS-EEG2 uses, resamples to 100 Hz, and arranges trials in THINGS-EEG2's image order with each repetition's
session. It first proves the image mapping from the metadata, or stops. The only number computed from the EEG is
each person's noise ceiling, needed for the P-6 margin. Output: the dataset `cross-subject-brain-encoding-alljoined`
(non-commercial research licence; share it only with accounts that accepted the Alljoined terms).

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09k-alljoined', 'cpu-build'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"people": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20], "repo": "Alljoined/Alljoined-1.6M", "slug": "cross-subject-brain-encoding-alljoined", "min_free_gb": 30, "ceiling_window_s": [0.1, 0.2], "ceiling_splits": 20, "seed": 13}')
import shutil
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

from cross_subject_encoding.data import alljoined as aj
from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval.noise_ceiling import noise_ceiling
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.eeg1_pilot import sha256_file
from cross_subject_encoding.utils.kaggle_export import export_folder

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")  # never printed or written
OUT, WORK = Path("/kaggle/tmp/alljoined_dataset"), Path("/kaggle/tmp/alljoined_download")
OUT.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUT).free / 1e9
if free_gb < FIXED["min_free_gb"]:
    raise RuntimeError(f"need about {FIXED['min_free_gb']} GB free under /kaggle/tmp, have {free_gb:.0f}")

def fetch(path):
    return Path(hf_hub_download(FIXED["repo"], path, repo_type="dataset", token=TOKEN, local_dir=WORK))

eeg2_meta = te.load_image_metadata(C.EEG_RAW if te.attached(C.EEG_RAW) else None)
# two candidate tables: the stimulus table (one row per image) and the first person's trial table; the mapping is
# taken from the first that proves it, and every table's columns are saved if none does
tables = {"stim_metadata": "preprocessed_eeg/experiment_metadata_categories.parquet",
          "sub-01 trials": "preprocessed_eeg/sub-01/experiment_metadata_categories.parquet"}
report, mapping, evidence, errors = {}, None, None, {}
for label, path in tables.items():
    table = pd.read_parquet(fetch(path))
    report[label] = {"columns": [str(c) for c in table.columns], "rows": int(len(table)),
                     "first_rows": table.head(3).astype(str).to_dict(orient="records")}
    print(label, "|", len(table), "rows | columns:", report[label]["columns"])
    print(table.head(3).to_string())
    try:
        mapping, evidence = aj.map_images(table.drop_duplicates(aj.path_column(table)) if aj.path_column(table) else table, eeg2_meta)
        evidence["table"] = label
        break
    except aj.MappingError as error:
        errors[label] = str(error)[:500]
(C.METRICS_DIR / "09k_schema.json").write_text(json.dumps({"tables": report, "errors": errors}, indent=1, default=str))
if mapping is None:
    raise RuntimeError("image mapping not proved from any table; see 09k_schema.json in the outputs")
report["mapping"] = evidence
counts = pd.Series([v[0] for v in mapping.values()]).value_counts().to_dict()
print("mapping:", evidence, counts)
assert counts.get("training") == aj.N_IMAGES["training"] and counts.get("test") == aj.N_IMAGES["test"], counts

In [ ]:
manifest_path = OUT / "09k_alljoined_manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {
    "source": "Alljoined-1.6M (Xu et al. 2025, arXiv:2508.18571), preprocessed release on Hugging Face; non-commercial research only",
    "format": "np.save of a dict: preprocessed_eeg_data (images, repetitions, 17, 100) float32, NaN where a repetition is missing; "
              "sessions (images, repetitions), -1 if missing; ch_names; times (0 to 0.99 s, 100 Hz). THINGS-EEG2 image order.",
    "mapping": evidence, "people": {}}
window = None
for s in FIXED["people"]:
    sub, t = f"sub-{s:02d}", time.monotonic()
    if sub in manifest["people"] and all((OUT / aj.FLAT.format(subject=sub, part=p)).exists() for p in ("training", "test")):
        print(sub, "already written")
        continue
    meta = pd.read_parquet(fetch(f"preprocessed_eeg/{sub}/experiment_metadata_categories.parquet"))
    missing = [c for c in ("partition", "session", "dropped") if c not in meta.columns] + ([] if aj.path_column(meta) else ["an image-path column"])
    if missing:
        raise RuntimeError(f"{sub}: metadata lacks {missing}; columns: {list(meta.columns)}")
    info = {}
    for part, name in (("test", "preprocessed_eeg_test_flat.npy"), ("training", "preprocessed_eeg_training_flat.npy")):
        path = fetch(f"preprocessed_eeg/{sub}/{name}")
        d = aj.load_flat(path)
        raw_shape, raw_names = list(np.shape(d["preprocessed_eeg_data"])), [str(c) for c in d["ch_names"]]
        x = aj.select_channels(np.asarray(d["preprocessed_eeg_data"]), raw_names)
        times = np.asarray(d["times"], np.float64)
        del d
        path.unlink()
        x, t100 = aj.to_100hz(x, times)
        if not np.isfinite(x).all():
            raise ValueError(f"{sub}/{part}: non-finite EEG")
        arr, sessions = aj.organize(x, aj.kept_rows(meta, part), mapping, part)
        del x
        n_reps = (~np.isnan(arr[:, :, 0, 0])).sum(1)
        info[part] = {"release_shape": raw_shape, "release_channels": len(raw_names), "release_times": [float(times[0]), float(times[-1])],
                      "shape": list(arr.shape), "reps_min": int(n_reps.min()), "reps_median": float(np.median(n_reps)),
                      "reps_max": int(n_reps.max()), "reps_counts": {int(k): int(v) for k, v in zip(*np.unique(n_reps, return_counts=True))},
                      "missing_share": float(1 - n_reps.sum() / arr.shape[0] / aj.MAX_REPS[part]),
                      "sessions": sorted(int(v) for v in np.unique(sessions) if v >= 0)}
        if part == "test":
            r = int(n_reps.min())
            window = (t100 >= FIXED["ceiling_window_s"][0]) & (t100 < FIXED["ceiling_window_s"][1])
            nc = noise_ceiling(arr[:, :r], FIXED["ceiling_splits"], FIXED["seed"], reps_scored=(r,))
            info[part].update({"ceiling_reps": r, "split_half_100_200ms": float(nc["r_split_half"][:, window].mean()),
                               f"max_r_{r}_100_200ms": float(nc[f"max_r_{r}"][:, window].mean())})
        np.save(OUT / aj.FLAT.format(subject=sub, part=part),
                {"preprocessed_eeg_data": arr, "sessions": sessions, "ch_names": list(aj.CHANNELS), "times": t100}, allow_pickle=True)
        info[part]["sha256"] = sha256_file(OUT / aj.FLAT.format(subject=sub, part=part))
        del arr
    manifest["people"][sub] = info
    manifest_path.write_text(json.dumps(manifest, indent=1))
    print(sub, f"{time.monotonic() - t:.0f} s | test reps min {info['test']['reps_min']}, ceiling split-half "
          f"{info['test']['split_half_100_200ms']:.3f} | training missing {info['training']['missing_share']:.1%} |",
          f"{(time.monotonic() - T0) / 60:.1f} min")
shutil.copy2(manifest_path, C.METRICS_DIR / manifest_path.name)
ceil = pd.DataFrame({s: {"split_half": v["test"]["split_half_100_200ms"], "test_reps_min": v["test"]["reps_min"],
                         "train_missing": v["training"]["missing_share"]} for s, v in manifest["people"].items()}).T
ceil.to_csv(C.METRICS_DIR / "09k_ceilings.csv")
print(ceil.round(3).to_string())
assert len(manifest["people"]) == len(FIXED["people"]), "not every person was written"

In [ ]:
url = export_folder(FIXED["slug"], "XSUB Alljoined in THINGS-EEG2 layout", OUT,  # Kaggle: 6-50 characters
                    message="Alljoined-1.6M, 20 people, 17 channels, 100 Hz")
print(url, "|", f"{(time.monotonic() - T0) / 60:.1f} min")